In [1]:
import polars as pl
import pandas as pd

import sys
sys.path.append('../04_utils')

from utils import name_finder, low_context_name_finder

In [2]:
# Define pathings
data_path = '../01_data/'
out_path = '../03_output/'

In [3]:
#Load data
verbs_df = pl.read_csv(data_path + 'DS1 verbs in -th.csv', separator= ',' )

verbs_df.head()

Left,KWIC,Right
str,str,str
"""of the forest graves. </s><s> …","""weareth""","""that ring, it allows for thine…"
"""''d, forsooth speak to me once…","""hath""","""spoken of thee. </s><s> I like…"
"""first set upon thee. </s><s> H…","""clingeth""","""to Shiva like some shadow...en…"
"""<s> Still I feel that boy hide…","""doth""","""bode most badly. </s><s> No re…"
""", plunge down from the plank, …","""seeketh""","""thee? </s><s> Why could thou n…"


In [4]:
#Load Character master table
char_master_df = pl.read_csv(data_path + 'das_char_master.csv')\
                   .with_columns(pl.col('Character').str.replace(',','').alias('Character'))

char_master_df.head()

Character,Class,Age
str,str,str
"""ALVINA OF THE DARKROOT WOOD""","""High""","""Old"""
"""ANASTACIA OF ASTORA""","""Low""","""Young"""
"""ANDRE OF ASTORA""","""Low""","""Old"""
"""BIG HAТ LOGAN""","""Low""","""Old"""
"""BLACKSMITH VAMOS""","""Low""","""Old"""


In [5]:
#Instantiate list with missing verbs
missing_verbs = ['Cometh','Taketh','Returneth','Hunteth','Trespasseth','Commenceth','Reneweth','Obscureth','Endeth','Serveth','Maketh',
                 'cometh','taketh','returneth','hunteth','trespasseth','commenceth','reneweth','obscureth','endeth','serveth','maketh']

unneeded_verbs = ['Bequeath','bequeath']

In [6]:
# Open raw text to look for missing verbs
with open(data_path + 'Ds corpus input.txt', 'r', encoding='utf-8') as file:
        raw_text = file.read()  # Read the entire content into a single string
        # print(raw_text)

In [7]:
#Construct empty dataframe to input results
left_context = []
verbs = []
right_context = []
#Search for not found verbs
for missing in missing_verbs:
    idx = 0
    context_list = raw_text.split(missing)
    while idx + 1 < len(context_list):
        left_context.append(context_list[idx])
        verbs.append(missing)
        right_context.append(context_list[idx + 1][:300])
        idx += 1

# Construct dataframe with the found verbs and contexts, enrich with character name and format contexts to facilitate legibility
missing_verbs_df = pl.DataFrame().with_columns(pl.Series(left_context).alias('Left'),
                                                 pl.Series(verbs).alias('KWIC'),
                                                 pl.Series(right_context).alias('Right'))\
                     .with_columns(pl.col('Left').map_elements(name_finder, return_dtype=pl.Utf8).alias('Character'))\
                     .with_columns(pl.col('Left').str.replace_all('<s>','').str.replace_all('</s>',''),
                                   pl.col('Right').str.replace_all('<s>','').str.replace_all('</s>',''))\
                     .with_columns(pl.col('KWIC').str.to_lowercase().alias('KWIC'))\
                     .with_columns(pl.col('Character').forward_fill().alias('Character'))

missing_verbs_df


Left,KWIC,Right,Character
str,str,str,str
"""NARRATOR Yes, indeed. The Dark…","""cometh""",""" soon. ""","""GIANT BLACKSMITH"""
""" soon. ""","""cometh""",""" soon. Mng. What's that? Shiny…","""GIANT BLACKSMITH"""
""" soon. Mng. What's that? Shiny…","""cometh""",""" thou to confess? Or to accuse…","""OSWALD OF CARIM"""
""" thou to confess? Or to accuse…","""cometh""",""" thou to confess? Or to aсcuse…","""OSWALD OF CARIM"""
"""NARRATOR Yes, indeed. The Dark…","""hunteth""",""" the enemies of the Lords, by …","""DARK SUN GWYNDOLIN"""
…,…,…,…
"""NARRATOR Yes, indeed. The Dark…","""endeth""",""" the Godmother, and now thou s…","""DARK SUN GWYNDOLIN"""
""" the Godmother, and now thou s…","""endeth""",""" this eternal twilight, and av…","""GWYNEVERE PRINCESS OF SUNLIGHT"""
""" this eternal twilight, and av…","""endeth""",""" this eternal twilight, and av…","""GWYNEVERE PRINCESS OF SUNLIGHT"""


In [8]:
missing_verbs_df.filter(pl.col('Character').is_null())

Left,KWIC,Right,Character
str,str,str,str


In [9]:
# Clean the raw text to facilitate matching with low_context_name_finder
punctuation = [',','.',';',':','!','?']

raw_text_clean = raw_text.replace('\n',' ').replace('…','...').replace('‘',"'").replace("' ","'").strip()
for punct in punctuation:
        raw_text_clean = raw_text_clean.replace(punct, punct + ' ').replace(punct,'')

raw_text_clean = raw_text_clean.split(' ')

raw_text_clean = [word for word in raw_text_clean if word != '']

raw_text_clean = ' '.join(raw_text_clean)

# raw_text_clean

In [10]:
# Extract the speaking character for each row, turn all KWIC lowercase to avoid redundant values.
verbs_df = verbs_df.filter(~pl.col('KWIC').is_in(missing_verbs)).filter(~pl.col('KWIC').is_in(unneeded_verbs))\
                   .with_columns(pl.col('Left').map_elements(lambda x: low_context_name_finder(x, raw_text_clean, 50), 
                                                             return_dtype=pl.Utf8).alias('Character'))\
                         .with_columns(pl.col('KWIC').str.to_lowercase().alias('KWIC'))\
                         .with_columns(pl.col('Left').str.replace_all('<s>','').str.replace_all('</s>','\n'),
                                       pl.col('Right').str.replace_all('<s>','').str.replace_all('</s>','\n'))\
                         

# Concat with the missing verbs and trim contexts to facilitate legibility.
verbs_df = pl.concat([verbs_df, missing_verbs_df])\
             .with_columns(pl.col('Left').str.tail(300).alias('Left'),
                           pl.col('Right').str.head(300).alias('Right'))

verbs_df

Left,KWIC,Right,Character
str,str,str,str
"""ould suit thee well. Hmm, I …","""weareth""","""that ring, it allows for thine…","""ALVINA OF THE DARKROOT WOOD"""
""" once more and will join us? …","""hath""","""spoken of thee. I like that …","""ALVINA OF THE DARKROOT WOOD"""
"""earn thee many more Perchance.…","""clingeth""","""to Shiva like some shadow...en…","""ALVINA OF THE DARKROOT WOOD"""
"""ll use us badly... Yet on gu…","""doth""","""bode most badly. No rest wil…","""ALVINA OF THE DARKROOT WOOD"""
"""shall be requited not. Thou …","""seeketh""","""thee? Why could thou not let…","""CROSSBREED PRISCILLA"""
…,…,…,…
"""he Darkmoon trespasseth upon t…","""endeth""",""" the Godmother, and now thou s…","""DARK SUN GWYNDOLIN"""
"""sen Undead. Come hither, child…","""endeth""",""" this eternal twilight, and av…","""GWYNEVERE PRINCESS OF SUNLIGHT"""
"""ire of our world. A grave and …","""endeth""",""" this eternal twilight, and av…","""GWYNEVERE PRINCESS OF SUNLIGHT"""


In [11]:
verbs_df.filter(pl.col('Character').is_null())#['Left'].to_list()

Left,KWIC,Right,Character
str,str,str,str


In [12]:
#Add character class and age information from master table
verbs_df = verbs_df.join(char_master_df, on='Character', how = 'left')

verbs_df

Left,KWIC,Right,Character,Class,Age
str,str,str,str,str,str
"""ould suit thee well. Hmm, I …","""weareth""","""that ring, it allows for thine…","""ALVINA OF THE DARKROOT WOOD""","""High""","""Old"""
""" once more and will join us? …","""hath""","""spoken of thee. I like that …","""ALVINA OF THE DARKROOT WOOD""","""High""","""Old"""
"""earn thee many more Perchance.…","""clingeth""","""to Shiva like some shadow...en…","""ALVINA OF THE DARKROOT WOOD""","""High""","""Old"""
"""ll use us badly... Yet on gu…","""doth""","""bode most badly. No rest wil…","""ALVINA OF THE DARKROOT WOOD""","""High""","""Old"""
"""shall be requited not. Thou …","""seeketh""","""thee? Why could thou not let…","""CROSSBREED PRISCILLA""","""High""","""Old"""
…,…,…,…,…,…
"""he Darkmoon trespasseth upon t…","""endeth""",""" the Godmother, and now thou s…","""DARK SUN GWYNDOLIN""","""High""","""Old"""
"""sen Undead. Come hither, child…","""endeth""",""" this eternal twilight, and av…","""GWYNEVERE PRINCESS OF SUNLIGHT""","""High""","""Old"""
"""ire of our world. A grave and …","""endeth""",""" this eternal twilight, and av…","""GWYNEVERE PRINCESS OF SUNLIGHT""","""High""","""Old"""


In [13]:
verbs_df.filter(pl.col('Character').is_null())

Left,KWIC,Right,Character,Class,Age
str,str,str,str,str,str


In [14]:
# Sanity check for extracted characters
verbs_df.to_pandas()['Character'].unique()

array(['ALVINA OF THE DARKROOT WOOD', 'CROSSBREED PRISCILLA',
       'DARK SUN GWYNDOLIN', 'DARKSTALKER KAATHЕ', 'DUSK OF OOLACILE',
       'GIANT BLACKSMITH', 'GWYNEVERE PRINCESS OF SUNLIGHT',
       'HAWKEYE GOUGН', 'OSCAR OF ASTORA', 'OSWALD OF CARIM'],
      dtype=object)

In [16]:
# Save enriched dataframe into a csv file.
verbs_df.write_csv(out_path + 'th_verbs_analysis.csv', separator= ';')